In [14]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

In [15]:
load_dotenv()

True

![Diagram](./images/00_Basic_RAG_Pipeline.png)

# Data Extraction

The first step in any RAG pipeline is to get the data you want to work with. Here we're building a customer-support assistant for **ShopEasy**, a multi-category e-commerce platform.

Our knowledge base lives in `shopeasy_knowledge_base.json` — 51 internal support documents across five types (`runbook`, `past_ticket`, `product_doc`, `bug_report`, `faq`) spanning five product areas (`payments`, `returns`, `shipping`, `orders`, `account`).

Each entry has a `content` string and a `metadata` object (`doc_type`, `product_area`, `priority`, `platform`, `customer_tier`, `status`, `title`). We load the JSON and wrap each entry in a LangChain `Document`, carrying the metadata along so it's available at retrieval time.

![Data Extraction](./images/01_data_extraction.png)

In [16]:
# Load the ShopEasy knowledge base from JSON and wrap each entry in a Document.
with open("shopeasy_knowledge_base.json") as f:
    kb = json.load(f)

docs = [
    Document(page_content=entry["content"], metadata=entry["metadata"])
    for entry in kb
]

In [17]:
len(docs) # Number of documents loaded from the ShopEasy knowledge base

51

In [18]:
# Let's inspect the metadata of one of the loaded documents.
# Each support doc carries structured metadata (doc_type, product_area, priority,
# platform, customer_tier, status, title) that we'll later use for filtering.

# docs[0].page_content
docs[0].metadata

{'doc_type': 'runbook',
 'product_area': 'payments',
 'priority': 'P1',
 'platform': 'all',
 'customer_tier': 'all',
 'status': 'active',
 'title': 'Handling Payment Declined Errors (PAY_DECLINED_402)'}

In [19]:
# To get a better sense of the content we've loaded,
# this code block defines a helper function wrap_text to format the text and then prints the content of each document.

def wrap_text(text, width=80):
    return '\n'.join([text[i:i+width] for i in range(0, len(text), width)])

for doc in docs:
    print(wrap_text(doc.page_content))
    print("-"*100)

Handling Payment Declined Errors (PAY_DECLINED_402)

Symptoms:
Customer reports 
that their payment was declined at checkout. The error page shows code PAY_DECLI
NED_402 with the message "Your payment could not be processed." The customer may
 say their card works fine on other websites.

Common Causes:
1. The card issuer
 declined the transaction due to insufficient funds, spending limits, or fraud d
etection triggers.
2. The billing address entered at checkout does not match the
 address on file with the card issuer (AVS mismatch).
3. The card has expired or
 the CVV entered is incorrect.
4. The customer's bank is blocking international 
transactions (ShopEasy processes payments through our EU-based payment gateway f
or some regions).
5. The customer is using a prepaid or virtual card that does n
ot support recurring authorizations (relevant for Subscribe & Save orders).

Res
olution Steps:
1. Ask the customer to verify the card details: number, expiratio
n date, CVV, and billing add

# Chunk the data

Now that we have our documents, the next step is to split them into smaller chunks. This is important for a few reasons:

__Vector search efficiency__: Smaller chunks are easier to search and retrieve.

__Context window limitations__: LLMs have a limited context window, so we need to make sure the retrieved information fits within that window.

__Relevance__: Smaller chunks are more likely to be focused on a specific topic, which improves the relevance of the retrieved information.

We'll use the `RecursiveCharacterTextSplitter` to split our documents. This splitter tries to split text on a series of characters (like newlines, spaces, etc.) in a recursive manner.

- `chunk_size=500`: This sets the maximum size of each chunk to 500 characters. Our support docs are fairly short, so a smaller chunk size means the longer runbooks and tickets actually split into several focused chunks instead of staying as one big blob.

- `chunk_overlap=100`: This creates an overlap of 100 characters between consecutive chunks. This helps to ensure that we don't lose any important context at the boundaries of our chunks.

- `add_start_index=True`: This will add the starting index of the chunk in the original document to the metadata.

Importantly, the metadata from each parent document (doc_type, product_area, title, …) is copied onto **every** chunk it produces, so we never lose track of where a chunk came from.

![Chunk the Data](./images/02_chunking.png)

In [20]:
# chunk the data

# ref: https://python.langchain.com/docs/concepts/text_splitters/
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # chunk size (characters)
    chunk_overlap=100,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

splits = text_splitter.split_documents(docs)

In [21]:
# See what chunking did: 51 documents become a larger number of smaller chunks.
print(f"Documents in:  {len(docs)}")
print(f"Chunks out:    {len(splits)}")
print("-" * 100)

# Find the longest document and show how it got split (with overlap + start_index).
longest_idx = max(range(len(docs)), key=lambda i: len(docs[i].page_content))
parent = docs[longest_idx]
print(f'Longest doc: "{parent.metadata["title"]}" ({len(parent.page_content)} chars)\n')

child_chunks = [
    s for s in splits if s.metadata.get("title") == parent.metadata["title"]
]
print(f"It split into {len(child_chunks)} chunks:\n")
for i, chunk in enumerate(child_chunks):
    start = chunk.metadata["start_index"]
    print(f"[chunk {i} | start_index={start} | {len(chunk.page_content)} chars]")
    print(chunk.page_content)
    print("-" * 100)

Documents in:  51
Chunks out:    193
----------------------------------------------------------------------------------------------------
Longest doc: "Troubleshooting Promo Code Failures (PROMO_INVALID_100)" (2179 chars)

It split into 6 chunks:

[chunk 0 | start_index=0 | 330 chars]
Troubleshooting Promo Code Failures (PROMO_INVALID_100)

Symptoms:
Customer reports that a promo code is not working at checkout. The error shown is either PROMO_INVALID_100 ("This promo code is not valid") or PROMO_EXPIRED_101 ("This promo code has expired"). Customer may insist they received the code via email or social media.
----------------------------------------------------------------------------------------------------
[chunk 1 | start_index=332 | 494 chars]
Common Causes:
1. The promo code has expired. All codes have an expiration date visible in Admin Panel > Promotions > [Code].
2. The code has a minimum order value that hasn't been met (e.g., "SAVE20 — $20 off orders $100+").
3. The code is r

In [22]:
for split in splits:
    print(wrap_text(split.page_content))
    print("-"*100)

Handling Payment Declined Errors (PAY_DECLINED_402)

Symptoms:
Customer reports 
that their payment was declined at checkout. The error page shows code PAY_DECLI
NED_402 with the message "Your payment could not be processed." The customer may
 say their card works fine on other websites.
----------------------------------------------------------------------------------------------------
Common Causes:
1. The card issuer declined the transaction due to insufficient f
unds, spending limits, or fraud detection triggers.
2. The billing address enter
ed at checkout does not match the address on file with the card issuer (AVS mism
atch).
3. The card has expired or the CVV entered is incorrect.
4. The customer'
s bank is blocking international transactions (ShopEasy processes payments throu
gh our EU-based payment gateway for some regions).
----------------------------------------------------------------------------------------------------
5. The customer is using a prepaid or virtual card th

In [23]:
splits[2].metadata

{'doc_type': 'runbook',
 'product_area': 'payments',
 'priority': 'P1',
 'platform': 'all',
 'customer_tier': 'all',
 'status': 'active',
 'title': 'Handling Payment Declined Errors (PAY_DECLINED_402)',
 'start_index': 738}

# Indexing

Now that we have our document chunks, we need to create an index that we can search. We'll use a vector store for this, which allows us to perform semantic search on our documents.

In this block, we're setting up our vector store using Pinecone and OpenAI embeddings.

- `embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)`: This initializes the embedding model that converts our document chunks into numerical vectors. `text-embedding-3-small` produces 1536-dimensional vectors by default, but the `text-embedding-3-*` models let you shorten the output via the `dimensions` parameter — here we use **512** to match our index.

- `pc = Pinecone(api_key=...)` / `index = pc.Index(...)`: This connects to Pinecone and grabs an **existing** index. Create the index in the [Pinecone console](https://app.pinecone.io/) first, making sure its dimension is **512** (to match the `dimensions=512` embeddings) and its metric is `cosine`.

- `vector_store = PineconeVectorStore(index=index, embedding=embeddings)`: This wraps the Pinecone index with our embedding model so we can add and search documents through LangChain. We store this notebook's vectors in their own `shopeasy-basic-rag` namespace.

This expects two environment variables: `PINECONE_API_KEY` and `PINECONE_INDEX_NAME` (plus `OPENAI_API_KEY` for the embeddings).

![Chunk the Data](./images/03_indexing.png)

<!-- ![Indexing](./images/03_indexing.png) -->

In [24]:
#Indexing

#define the embeddings model
#ref: https://python.langchain.com/docs/integrations/text_embedding/
#text-embedding-3-small defaults to 1536 dims, but the text-embedding-3-* models
#support shortening the output via `dimensions`. We use 512 to match the index.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)  # 512-dim vectors

#connect to an existing Pinecone index
#(create/manage the index in the Pinecone console; its dimension must be 512
# to match the `dimensions=512` embeddings above, with the cosine metric)
#ref: https://python.langchain.com/docs/integrations/vectorstores/pinecone/
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])

#define the vector store (this notebook's data lives in the "shopeasy-basic-rag" namespace)
#ref: https://python.langchain.com/docs/concepts/vectorstores/
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="shopeasy-basic-rag",
)

In [25]:
#Now, we add our document chunks to the vector store.
# This process will convert each chunk into a vector and store it in the database.

document_ids = vector_store.add_documents(documents=splits)
document_ids

['1adf7561-ba45-4125-a956-3326c043d95f',
 'f0f0f03f-55f7-4501-a3c0-dfcf46b4161a',
 '1658e5f2-2664-49c6-943b-7494b86861dd',
 'e3a7e215-2f03-424d-a383-1cce31dbdbff',
 '7690390a-0403-44ad-aa54-8dd44e43aeab',
 'e8efd969-7342-45cd-8bc3-bcbbfb75ed31',
 'c464b3ea-1b6f-4040-ba06-7b22c5592be5',
 '785733fc-7a99-425d-b11f-4e2f9005d871',
 '5ec4902c-dac8-4306-ae98-aae1f81132e2',
 '9bdb30c5-a792-4447-95a9-caef32bae998',
 'e28aa10e-7502-406c-9caf-38d601846f1d',
 '02942876-8d4f-4529-8e8d-a152f7eee5b6',
 '1b44a820-ec35-4cb8-82a5-a357a046530f',
 'dc7428b4-0d2e-4a65-bf03-87c0248b63bc',
 '341db1bb-231a-4976-a3dd-69d12c2a1bd1',
 'a277526f-6cd4-4059-9abc-a6ae22636993',
 '45fd1a6f-0788-45e2-aae5-f5f4bcf39dc9',
 'f19a007c-b533-4428-a024-d6e1ad40f713',
 'c09f7e58-3c00-46ff-9638-644360536107',
 'c7084a2a-452b-4c36-b0ba-44da534b1bc8',
 '275457eb-733f-4fba-83fc-32176a18464a',
 '7471099d-c3b5-45d9-b3f7-4bedee75ee7b',
 'c77e4ae2-9a19-4272-973a-5d23c5ffed46',
 '454fb610-924b-4fef-a6cc-1f4b1dc4a6e1',
 '0b829273-c51d-

In [28]:
# Let's retrieve a record from the Pinecone index by its ID to confirm it was indexed.
# Note: Pinecone is eventually consistent, so a just-added vector may take a
# moment to become fetchable.
index.fetch(ids=[document_ids[10]], namespace="shopeasy-basic-rag")

FetchResponse(namespace='shopeasy-basic-rag', vectors={'e28aa10e-7502-406c-9caf-38d601846f1d': Vector(id='e28aa10e-7502-406c-9caf-38d601846f1d', values=[-0.0434570312, -0.0151367188, 0.0249328613, 0.0202178955, -0.0328369141, 0.0157318115, 0.0121688843, -0.00724029541, -0.00577926636, 0.0324401855, -0.00758361816, 0.0747070312, -0.0734863281, -0.0370178223, -0.0582885742, 0.0565490723, 0.0238189697, -0.0185699463, -0.108276367, 0.0905761719, 0.0606689453, -0.0141983032, 0.0716552734, -0.0277099609, -0.0503540039, 0.0354614258, -0.00410842896, -0.0685424805, 0.0392456055, -0.0249481201, 0.0951538086, -0.0328369141, 0.0487976074, 0.0104980469, 0.0142745972, 0.0092086792, -0.0250701904, 0.0942993164, -0.024017334, 0.000823020935, -0.00131893158, -0.00396347046, -0.00626754761, 0.0269775391, -0.0223083496, 0.0203704834, -0.0372619629, -0.0397644043, -0.0229187012, 0.0393066406, 0.0057144165, 0.0835571289, -0.0922241211, 0.0530090332, 0.037902832, -0.013458252, 0.0520019531, 0.00694274902, 

# Retrieval

With our documents indexed, we can now perform retrieval. The goal of this step is to find the most relevant document chunks for a given customer issue.

First, we'll set up our LLM and a prompt template.

- `llm = ChatOpenAI(model="gpt-4.1-mini")`: We'll use OpenAI's `gpt-4.1-mini` model for generation.

- `template = ...`: This is the prompt template that combines the retrieved context with the customer's issue. The `{context}` and `{question}` placeholders are filled in at generation time.

![Retrieval](./images/04_retrieval.png)

In [29]:
#configure the llm
llm = ChatOpenAI(model="gpt-4.1-mini")

#set the prompt template
template = """You are a customer-support assistant for ShopEasy, an e-commerce platform.
Use the following pieces of retrieved internal knowledge-base context to help resolve the customer's issue.
If the context doesn't contain the answer, say you don't have that information rather than guessing.
Be concise and practical: state the likely cause and the next step the agent should take.

Context:
{context}

Customer issue: {question}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

In [30]:
# A sample customer support ticket (README Query 1).
user_question = "Customer says their order never arrived even though tracking shows delivered"

In [31]:
# Now, we'll use the vector store's similarity_search method to find the top 5 most similar chunks to the customer issue.
retrieved_docs = vector_store.similarity_search(user_question, k=5)

In [32]:
# Review the retrieved chunks. We print the metadata header so we can SEE the noise:
# semantic search will surface relevant shipping docs, but may also pull in
# returns/refund docs because the language overlaps.
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[past_ticket / shipping] Order ORD-88921 shows delivered but never received
Ticket #T-10234 — Customer's order ORD-88921 shows delivered but never received

Customer reported: "My order was marked as delivered two days ago but I never got it. I was home all day. The tracking says it was left at the front door but nothing was there."

Investigation: Checked tracking via FedEx — GPS coordinates of the delivery scan were 0.3 miles from the customer's address. Likely misdelivered to a nearby address by the driver.
----------------------------------------------------------------------------------------------------
[runbook / shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
Investigating Shipping Delays (SHIP_DELAYED_301)

Symptoms:
Customer reports their order has not arrived by the estimated delivery date. The tracking page may show SHIP_DELAYED_301 or the tracking status has not updated in 48+ hours. Customer may be frustrated especially if they paid for expedited shipping.
----

In [33]:
# The similarity_search_with_score method returns the chunks along with their similarity scores.
vector_store.similarity_search_with_score(user_question, k=5)

[(Document(id='b3ec1440-8fbc-45b9-9282-bcf59e5703f0', metadata={'customer_tier': 'regular', 'doc_type': 'past_ticket', 'platform': 'web', 'priority': 'P1', 'product_area': 'shipping', 'start_index': 0.0, 'status': 'resolved', 'title': 'Order ORD-88921 shows delivered but never received'}, page_content='Ticket #T-10234 — Customer\'s order ORD-88921 shows delivered but never received\n\nCustomer reported: "My order was marked as delivered two days ago but I never got it. I was home all day. The tracking says it was left at the front door but nothing was there."\n\nInvestigation: Checked tracking via FedEx — GPS coordinates of the delivery scan were 0.3 miles from the customer\'s address. Likely misdelivered to a nearby address by the driver.'),
  0.68602407),
 (Document(id='e28aa10e-7502-406c-9caf-38d601846f1d', metadata={'customer_tier': 'all', 'doc_type': 'runbook', 'platform': 'all', 'priority': 'P1', 'product_area': 'shipping', 'start_index': 0.0, 'status': 'active', 'title': 'Invest

# Generation

The final step is to use the retrieved documents to generate an answer to the customer's issue.

We'll combine the content of the retrieved documents and use our prompt template to create a final prompt for the LLM.


![Generation](./images/05_generation.png)

In [34]:
#generate answer

docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
response = llm.invoke(prompt)

In [35]:
#generated response
response.content

"The likely cause is a misdelivery, as the delivery scan GPS was 0.3 miles from the customer's address indicating the package was left at a nearby location by mistake.\n\nNext step: Advise the customer you are escalating the issue to the shipping carrier to locate the package or initiate a claim. Offer to send a replacement or refund if the package remains lost."

In [36]:
# We can also include citations in our response by extracting the source from the metadata of the retrieved docs.
# This dataset has no "source" URL, so we cite each doc by its title and doc_type instead.

sources = [f"{doc.metadata['title']} ({doc.metadata['doc_type']})" for doc in retrieved_docs]

print(f"Sources: {sources}\n\n")
print(f'Answer: {response.content}')

Sources: ['Order ORD-88921 shows delivered but never received (past_ticket)', 'Investigating Shipping Delays (SHIP_DELAYED_301) (runbook)', 'Refund not received after 10 business days for ORD-91045 (past_ticket)', 'Tracking number TRK-FX-998834 shows no movement for 5 days (past_ticket)', 'Handling Wrong Item Received (runbook)']


Answer: The likely cause is a misdelivery, as the delivery scan GPS was 0.3 miles from the customer's address indicating the package was left at a nearby location by mistake.

Next step: Advise the customer you are escalating the issue to the shipping carrier to locate the package or initiate a claim. Offer to send a replacement or refund if the package remains lost.


### Langchain Retreiver

LangChain provides a `Retriever` interface, which is a more general way to retrieve documents. A vector store can be used as the backbone of a retriever, but there are other types of retrievers as well.

Here, we're creating a retriever from our vector store. We can also specify search arguments like k (the number of documents to retrieve) and search_type.

In [37]:
# Ref: https://python.langchain.com/docs/concepts/retrievers/
retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')

retrieved_docs = retriever.invoke(user_question)

for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[past_ticket / shipping] Order ORD-88921 shows delivered but never received
Ticket #T-10234 — Customer's order ORD-88921 shows delivered but never received

Customer reported: "My order was marked as delivered two days ago but I never got it. I was home all day. The tracking says it was left at the front door but nothing was there."

Investigation: Checked tracking via FedEx — GPS coordinates of the delivery scan were 0.3 miles from the customer's address. Likely misdelivered to a nearby address by the driver.
----------------------------------------------------------------------------------------------------
[runbook / shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
Investigating Shipping Delays (SHIP_DELAYED_301)

Symptoms:
Customer reports their order has not arrived by the estimated delivery date. The tracking page may show SHIP_DELAYED_301 or the tracking status has not updated in 48+ hours. Customer may be frustrated especially if they paid for expedited shipping.
----

![Overall](./images/06_put_it_together.png)

In [38]:
# Now, let's put it all together in a single function.
def generate_answer(user_question):
    #retrieve the relevant docs
    retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')
    retrieved_docs = retriever.invoke(user_question)

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return response.content

generate_answer("Customer says their order never arrived even though tracking shows delivered")

"The likely cause is a misdelivery, as the GPS coordinates show the package was left about 0.3 miles from the customer's address. The next step is to inform the customer that the package was likely delivered to a nearby address by mistake and to check with neighbors or nearby residences. Meanwhile, escalate the issue to ShopEasy's delivery team and FedEx to initiate a package trace or recovery."

### More sample tickets — watch the noise

Plain semantic search handles natural language well, but when a query spans multiple product areas it casts a wide net. Run these and inspect the `[doc_type / product_area]` headers to see shipping, payments, and returns docs all mixed together — exactly the problem that metadata filtering (Notebook 2) and hybrid search (Notebook 3) will fix.

In [39]:
# Query 2: a broad query returns broadly relevant results — returns runbook, return
# policy, return FAQ, plus past return tickets... and maybe warranty/billing noise.
generate_answer("Customer wants to return a product they bought last month")

"Please verify if the product is within the return window by checking the delivery date in the Admin Panel under Orders. If it's electronics, the return window is 15 days; for other categories, typically 30 days. If the item is eligible and defective, initiate a return with reason code RETURN_DEFECTIVE. If outside the return window but still under the 1-year warranty, confirm it's a manufacturing defect before proceeding."

In [40]:
# Query 3: an issue that spans payments + shipping + returns at once — semantic
# search casts a wide net across all three areas, making it hard to focus.
generate_answer("Customer is upset about being charged but never receiving their product")

'The likely cause is a payment authorization hold or delay in refund processing despite the order showing as "Cancelled." The next step is to verify if the charge is only a pending authorization or a posted payment. Then, escalate the issue to the billing team to confirm the refund status and ensure the $67.89 is returned promptly. Inform the customer you are investigating and will update them shortly.'